# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR² Dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant Metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Authors: {[auth['@id'] for auth in getattr(metadata,'author',[])]}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview

Review available record sets, including their IDs and fields.

Here, we print the available RecordSets in the dataset and their fields using their `@id` attributes. This helps determine which data can be loaded and analyzed.

In [ ]:
# List available RecordSets and Fields for exploration
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print("Available RecordSets:")
    for i, record_set in enumerate(metadata.recordSet):
        rec_id = record_set['@id'] if isinstance(record_set, dict) else record_set
        print(f"  {i+1}. RecordSet @id: {rec_id}")
        # Load actual record set object
        rec_set_obj = dataset._find(rec_id)
        # List associated fields
        if hasattr(rec_set_obj, 'field'):
            print("     Fields:")
            for field in rec_set_obj.field:
                field_id = field['@id'] if isinstance(field, dict) else field
                field_obj = dataset._find(field_id)
                print(f"       - {field_id} (name: {getattr(field_obj,'name',field_id)})")
else:
    print("No record sets found in the metadata. The schema may provide records through distributions.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

> **Note:** All operations reference entities by their `@id` fields, which is recommended for reproducibility and aligning with FAIR data standards.

If the RecordSets are not present via the Croissant schema's `recordSet` key, you can often access records using the main set provided by the dataset (even when only one is present).

In [ ]:
# Step 1: Gather the @id(s) of the record sets for extraction
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = [rs['@id'] if isinstance(rs, dict) else rs for rs in metadata.recordSet]
else:
    # Heuristic: Try using the first distribution as a record set (common for single-table datasets)
    if hasattr(metadata, 'distribution') and metadata.distribution:
        dist_id = metadata.distribution[0]['@id'] if isinstance(metadata.distribution[0], dict) else metadata.distribution[0]
        record_sets = [dist_id]
        print(f"Record set inferred from distribution: {dist_id}")
    else:
        print("No distributions or record sets available.")
        record_sets = []

# Step 2: Load records into DataFrames
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head(3))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For further analysis, choose the first record_set
main_record_set_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering numeric values, normalizing, and grouping by attributes. In all cases, columns are referenced by their field or column `@id` when possible.

Below, select a numeric field (e.g., protein abundance, peptide counts, molecular weight) and a group field (e.g., description or accession) for demonstration.

In [ ]:
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Inspect columns to pick appropriate numeric and group fields
    print(f"Columns available for EDA: {df.columns.tolist()}")
    # Heuristically select numeric fields
    # Attempt to pick peptide count, molecular weight, or a normalized abundance column by likely names
    possible_numeric = [c for c in df.columns if any(sub in c.lower() for sub in ['abundance', 'count', 'molecular', 'mass', 'mw', 'coverage', 'peptide'])]
    numeric_field = possible_numeric[0] if possible_numeric else None
    group_field = None
    # Pick something to group on, e.g., 'description', 'accession', or first string field
    for c in df.columns:
        if c != numeric_field and df[c].dtype == 'object':
            group_field = c
            break
    print(f"Using numeric_field='{'None' if numeric_field is None else numeric_field}' and group_field='{'None' if group_field is None else group_field}' for examples.")

    # Try filtering (e.g., peptide count > 10)
    threshold = 10
    if numeric_field and numeric_field in df.columns:
        try:
            filtered_df = df.copy()
            # Convert numeric_field to float
            filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
            filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold} (showing top 5):")
            print(filtered_df.head())

            # Normalization
            norm_col = f"{numeric_field}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"\nNormalized {numeric_field} for filtered records (top 5):")
            print(filtered_df[[numeric_field, norm_col]].head())

            # Grouping
            if group_field and group_field in filtered_df.columns:
                grouped_df = (
                    filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                )
                print(f"\nGrouped mean {numeric_field} by {group_field} (top 5):")
                print(grouped_df.head())
        except Exception as ex:
            print(f"Could not perform EDA due to error: {ex}")
    else:
        print("No numeric field could be found for EDA.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization

Visualize data distributions or relationships between numeric fields. Below is an example using matplotlib. If no numeric field is available, this section will not plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and 'filtered_df' in locals() and not filtered_df.empty and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        # Show mean by group if group_field is not too unique
        value_counts = filtered_df[group_field].value_counts()
        top_groups = value_counts.index[:5].tolist()  # Avoid labels overflow
        group_means = filtered_df.groupby(group_field)[numeric_field].mean().loc[top_groups]
        group_means.plot(kind='bar', title=f"Mean {numeric_field} by {group_field} (Top 5)", ylabel=f"Mean {numeric_field}")
        plt.show()
else:
    print("Not enough numeric data found or no filtered DataFrame available for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded and explored the FAIR² mass spectrometry extracellular vesicle dataset using the `mlcroissant` library,
- Examined available record sets and fields by their Croissant `@id` values,
- Extracted records into pandas DataFrames for practical manipulation,
- Applied EDA techniques—such as filtering, normalization, and aggregation—referencing all entities by their `@id`,
- Generated visualizations to summarize numeric field distributions and group statistics.

This workflow can be adapted for other Croissant-compliant datasets by adjusting the record set and field `@id` variables.

**Next steps:** Consider using the processed data for additional machine learning analyses, or join with external protein annotations for richer biological insights.